In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import geopandas as gp
import libpysal as ps

from mgwr.sel_bw import Sel_BW
from mgwr.gwr import GWR as OfficialGWR

from src.log.gwr_logger import GwrLogger
from src.optimizer.gwr_optimizer import GwrOptimizer
from src.kernel.gwr_kernel import GwrKernel
from src.model.gwr import GWR
from src.dataset.interfaces.idataset import FieldInfo
from src.dataset.spatial_dataset import SpatialDataset


In [7]:
# Create a logger to record the GWR model's information.
logger = GwrLogger()

georgia_data = pd.read_csv(ps.examples.get_path('GData_utm.csv'))
dataset = SpatialDataset(
    georgia_data,
    FieldInfo(
        predictor_fields=['PctFB', 'PctBlack', 'PctRural'],
        response_field='PctBach',
        coordinate_x_field='X',
        coordinate_y_field='Y'
    ),
    logger=logger,
    isSpherical=False
)

{'2025-09-24 15:45:44': 'SpatialDataset : Data schema is verified.'}


In [8]:
# Create a GWR kernel and GWR model.
kernel = GwrKernel(dataset, 'bisquare')
gwr = GWR(dataset, kernel, logger)

# Use the bandwidth optimizer to automatically find the optimal bandwidth.
optimizer = GwrOptimizer(gwr, kernel, logger)
optimal_bandwidth = optimizer.optimize()


{'2025-09-24 15:45:44': 'GWR : GWR model is initialized.'}
{'2025-09-24 15:45:44': 'GwrOptimizer : Bandwidth 107.29490168751578, AICc 299.75802040708595, R2 0.6826497327135301'}
{'2025-09-24 15:45:45': 'GwrOptimizer : Bandwidth 142.7050983124842, AICc 302.17466423538303, R2 0.6585140155210543'}
{'2025-09-24 15:45:45': 'GwrOptimizer : Bandwidth 85.41019662496845, AICc 303.1797562723929, R2 0.6939013951017431'}
{'2025-09-24 15:45:45': 'GwrOptimizer : Bandwidth 120.8203932499369, AICc 299.6170948047427, R2 0.6751807671965796'}
{'2025-09-24 15:45:46': 'GwrOptimizer : Bandwidth 129.17960675006307, AICc 299.8717241663784, R2 0.6699388552526588'}
{'2025-09-24 15:45:46': 'GwrOptimizer : Bandwidth 115.65411518764195, AICc 299.0840712206579, R2 0.6790141917452021'}
{'2025-09-24 15:45:46': 'GwrOptimizer : Bandwidth 112.46117974981073, AICc 299.2869991564815, R2 0.6804219570368577'}
{'2025-09-24 15:45:46': 'GwrOptimizer : Bandwidth 117.62745781210567, AICc 299.0508086830287, R2 0.678074266959346'}

In [9]:
g_X = dataset.X[:, 1:]
g_y = dataset.y.reshape(-1, 1)
g_coords = dataset.coordinates.tolist()


g_X = dataset.X
g_y = dataset.y
g_coords = dataset.coordinates.tolist()

gwr_selector = Sel_BW(g_coords, g_y, g_X)
gwr_bw = gwr_selector.search(bw_min=2)
official_gwr = OfficialGWR(g_coords, g_y, g_X, gwr_bw).fit()

In [15]:
gwr.update_bandwidth(gwr_bw).fit()
official_gwr = OfficialGWR(g_coords, g_y, g_X, gwr_bw).fit()

In [16]:
print("Custom GWR R2:", gwr.r_squared)
print("Official GWR R2:", official_gwr.R2)

Custom GWR R2: 0.678074266959346
Official GWR R2: 0.678074266959346


In [18]:
print(gwr.aic)
print(gwr.aicc)
print(official_gwr.aic)
print(official_gwr.aicc)

296.61592295199813
299.0508086830287
296.6159229519982
299.0508086830288


In [23]:
print(gwr.betas[0])
print(official_gwr.params[0])

[-0.23204579  0.22820815  0.05697445 -0.42649461]
[-0.23204579  0.22820815  0.05697445 -0.42649461]


In [24]:
print(gwr.kernel.bandwidth)
print(gwr_bw)

117.0
117.0
